In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

In [ ]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [ ]:
df

In [ ]:
df.sample(5)

In [ ]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [ ]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [ ]:
train_y.value_counts()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [ ]:
train_x.shape

In [ ]:
train_x_vector.shape

In [ ]:
type(train_x_vector)

In [ ]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [ ]:
primera_resenia

In [ ]:
train_x.iloc[0]

In [ ]:
primera_resenia[primera_resenia != 0]

In [ ]:
train_x.iloc[0]

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

In [ ]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

In [ ]:
print(svc.score(test_x_vector, test_y))

In [ ]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

In [ ]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

## Comparación de modelos adicionales

En esta sección se compara el modelo SVM ya entrenado con:
- LogisticRegression
- GaussianNB
- DecisionTreeClassifier

Consideraciones metodológicas importantes:
- SVM, LogisticRegression y DecisionTreeClassifier usan directamente los vectores TF-IDF (`train_x_vector` y `test_x_vector`).
- GaussianNB no acepta matrices sparse de `scipy`.
- Para evitar convertir toda la matriz TF-IDF a densa (riesgo de memoria), se aplica `TruncatedSVD` únicamente para GaussianNB.
- Por lo tanto, GaussianNB se evalúa con una representación densa reducida, no con exactamente la misma representación dimensional que los otros tres modelos.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
def obtener_metricas(y_real, y_pred):
    return {
        'accuracy': accuracy_score(y_real, y_pred),
        'precision': precision_score(y_real, y_pred, pos_label='positive', zero_division=0),
        'recall': recall_score(y_real, y_pred, pos_label='positive', zero_division=0),
        'f1_score': f1_score(y_real, y_pred, pos_label='positive', zero_division=0)
    }


comparacion_resultados = {}

# Usamos el SVM ya entrenado, sin reentrenarlo ni cambiar su configuración.
svm_pred = svc.predict(test_x_vector)
comparacion_resultados['SVM'] = obtener_metricas(test_y, svm_pred)

print('Matriz de confusión - SVM')
print(confusion_matrix(test_y, svm_pred, labels=['positive', 'negative']))

In [ ]:
# LogisticRegression (usa directamente TF-IDF sparse)
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(train_x_vector, train_y)

log_reg_pred = log_reg.predict(test_x_vector)
comparacion_resultados['LogisticRegression'] = obtener_metricas(test_y, log_reg_pred)

print('Matriz de confusión - LogisticRegression')
print(confusion_matrix(test_y, log_reg_pred, labels=['positive', 'negative']))

# DecisionTreeClassifier (usa directamente TF-IDF sparse)
dec_tree = DecisionTreeClassifier(random_state=42)
dec_tree.fit(train_x_vector, train_y)

dec_tree_pred = dec_tree.predict(test_x_vector)
comparacion_resultados['DecisionTreeClassifier'] = obtener_metricas(test_y, dec_tree_pred)

print('\nMatriz de confusión - DecisionTreeClassifier')
print(confusion_matrix(test_y, dec_tree_pred, labels=['positive', 'negative']))

In [ ]:
# GaussianNB requiere representación densa.
# Para evitar convertir toda la matriz TF-IDF a densa, aplicamos SVD solo aquí.
svd_components = min(300, train_x_vector.shape[1] - 1)
svd = TruncatedSVD(n_components=svd_components, random_state=42)

train_x_gnb = svd.fit_transform(train_x_vector)
test_x_gnb = svd.transform(test_x_vector)

gnb = GaussianNB()
gnb.fit(train_x_gnb, train_y)

gnb_pred = gnb.predict(test_x_gnb)
comparacion_resultados['GaussianNB'] = obtener_metricas(test_y, gnb_pred)

print('Matriz de confusión - GaussianNB (con TruncatedSVD)')
print(confusion_matrix(test_y, gnb_pred, labels=['positive', 'negative']))

In [ ]:
tabla_comparativa = pd.DataFrame(comparacion_resultados).T
tabla_comparativa = tabla_comparativa[['accuracy', 'precision', 'recall', 'f1_score']]
tabla_comparativa = tabla_comparativa.round(4)

print('Tabla comparativa de modelos')
tabla_comparativa

## Conclusión breve

La siguiente celda genera una conclusión automática basada en la tabla comparativa obtenida en la ejecución actual.

Nota: la comparación de GaussianNB debe interpretarse considerando que usa una representación reducida con `TruncatedSVD`, mientras que SVM, LogisticRegression y DecisionTreeClassifier usan directamente TF-IDF sparse.

In [ ]:
mejor_f1 = tabla_comparativa['f1_score'].idxmax()
mejor_accuracy = tabla_comparativa['accuracy'].idxmax()

print('Conclusión automática basada en esta ejecución:')
print(f'- El mejor modelo según F1-score fue: {mejor_f1}.')
print(f'- El mejor modelo según accuracy fue: {mejor_accuracy}.')
print('- Recomendación académica: priorizar la métrica más adecuada al objetivo del problema (por ejemplo, F1-score cuando interesa balancear precisión y recall).')
print('- Recordatorio metodológico: GaussianNB fue evaluado con reducción de dimensionalidad (TruncatedSVD), por lo que su comparación no es sobre la misma representación dimensional original de TF-IDF.')